# 05 — Selecting the Best Technique and Comparing Models Using BERTScore F1

This notebook uses **BERTScore F1 only** to compare `zero-shot`, `one-shot`, and `few-shot`, first within each model and then across models.

## Input

Place the BERTScore CSV files in:

```text
bertscore/
```

The notebook identifies the **model** and **technique** from the CSV content (`modelo/model` and `tecnica/technique`), not from the filename.

Expected design:

- 5 models;
- 3 techniques;
- 259 cases per condition;
- 10 executions per case;
- 15 BERTScore files;
- 38,850 raw scores.

## Pipeline

1. Validate files and executions.
2. Aggregate the 10 executions by `case × model × technique`.
3. Calculate mean BERTScore, median, SD, and CV.
4. Diagnose normality of paired differences.
5. Compare techniques within each model:
   - Friedman;
   - paired Wilcoxon;
   - Holm;
   - rank-biserial;
   - 95% bootstrap CI of the mean difference.
6. Identify the best technique for each model.
7. Identify the best global technique.
8. Fit a mixed linear model `BERTScore ~ Model * Technique + (1|Case)`.
9. Compare models:
   - using the best observed configuration of each model;
   - under the same global technique.
10. Generate a **complete TXT summary**.

The 10 executions are aggregated before inference to avoid pseudoreplication. Stability continues to be evaluated through the CV of the 10 executions.


In [ ]:
from pathlib import Path
from itertools import combinations
import re, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

try:
    import statsmodels.formula.api as smf
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False

## 1. Configuration


In [ ]:
BERTSCORE_DIR = Path("bertscore")
OUTPUT_DIR = Path("resultados_bertscore")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_MODEL_COUNT = 5
EXPECTED_TECHNIQUES = ["zero-shot", "one-shot", "few-shot"]
EXPECTED_TOTAL_CASES = 259
EXPECTED_EXECUTIONS = set(range(1, 11))
EXPECTED_TOTAL_EXECUTIONS = 10
EXPECTED_ROWS_PER_FILE = EXPECTED_TOTAL_CASES * EXPECTED_TOTAL_EXECUTIONS
EXPECTED_BERTSCORE_FILES = EXPECTED_MODEL_COUNT * len(EXPECTED_TECHNIQUES)

ALPHA = 0.05
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260828
STRICT_VALIDATION = True

## 2. Helper Functions


In [ ]:
def normalize_column_name(effect_name):
    effect_name = str(effect_name).strip().lower()
    for a,b in {"á":"a","à":"a","ã":"a","â":"a","é":"e","ê":"e","í":"i","ó":"o","ô":"o","õ":"o","ú":"u","ç":"c"}.items():
        effect_name = effect_name.replace(a,b)
    return re.sub(r"[^a-z0-9]+","_",effect_name).strip("_")

def normalize_technique(v):
    v = re.sub(r"\s+","-",str(v).strip().lower().replace("_","-"))
    base = v.replace("-","")
    mapping = {"zeroshot":"zero-shot","oneshot":"one-shot","fewshot":"few-shot"}
    return mapping.get(base, v)

def read_csv_robust(path):
    last_error = None
    for enc in ["utf-8-sig","utf-8","latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            if len(df.columns)==1:
                df2 = pd.read_csv(path, sep=";", encoding=enc)
                if len(df2.columns)>1: df=df2
            return df
        except Exception as e:
            last_error=e
    raise RuntimeError(f"Failed to read {path}: {last_error}")

def standardize_columns(df):
    df=df.copy()
    df.columns=[normalize_column_name(c) for c in df.columns]
    df=df.rename(columns={"modelo":"model","tecnica":"technique","execucao":"execution"})
    return df

def unique_value(df,column,file_name):
    if column not in df.columns:
        raise ValueError(f"{file_name}: missing column: {column}")
    values=df[column].dropna().astype(str).str.strip().unique()
    if len(values)!=1:
        raise ValueError(f"{file_name}: expected 1 value in {column}, found {values[:10]}")
    return values[0]

def holm_bonferroni(p_values):
    p=np.asarray(p_values,dtype=float)
    out=np.full(len(p),np.nan)
    ids=np.where(np.isfinite(p))[0]
    if not len(ids): return out
    pv=p[ids]; order=np.argsort(pv); sorted_p=pv[order]; m=len(sorted_p)
    adj=np.empty(m); previous=0.0
    for i,value in enumerate(sorted_p):
        current=min(1.0,(m-i)*value)
        previous=max(previous,current); adj[i]=previous
    reverse=np.empty(m); reverse[order]=adj; out[ids]=reverse
    return out

def safe_shapiro(values):
    s=pd.Series(values)
    s=pd.to_numeric(s,errors="coerce").replace([np.inf,-np.inf],np.nan).dropna()
    if len(s)<3 or s.nunique()<2:
        return {"n":len(s),"W":np.nan,"p":np.nan,"rejeita":None}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r=stats.shapiro(s.to_numpy(float))
    return {"n":len(s),"W":float(r.statistic),"p":float(r.pvalue),"rejeita":bool(r.pvalue<ALPHA)}

def paired_rank_biserial(a,b):
    d=np.asarray(a,float)-np.asarray(b,float)
    d=d[d!=0]
    if not len(d): return 0.0
    ranks=stats.rankdata(np.abs(d))
    wp=float(ranks[d>0].sum()); wn=float(ranks[d<0].sum())
    return (wp-wn)/(wp+wn) if (wp+wn) else 0.0

def bootstrap_mean_diff(a,b,n=N_BOOTSTRAP,seed=BOOTSTRAP_SEED):
    d=np.asarray(a,float)-np.asarray(b,float)
    rng=np.random.default_rng(seed)
    means=np.empty(n)
    for i in range(n):
        idx=rng.integers(0,len(d),len(d))
        means[i]=d[idx].mean()
    lo,hi=np.percentile(means,[2.5,97.5])
    return float(d.mean()),float(lo),float(hi)

## 3. Read, Identify, and Validate the CSV Files


In [ ]:
csv_files=sorted(BERTSCORE_DIR.rglob("*.csv"))
if not csv_files:
    raise RuntimeError(f"No CSV file found in {BERTSCORE_DIR.resolve()}")

inventory_rows=[]; data_by_config={}; ignored_files=[]

for path in csv_files:
    df=standardize_columns(read_csv_robust(path))
    if "bertscore_f1" not in df.columns:
        ignored_files.append(path.name)
        continue

    model=unique_value(df,"model",path.name)
    tech=normalize_technique(unique_value(df,"technique",path.name))

    for c in ["case_id","execution","generation_id"]:
        if c not in df.columns:
            raise ValueError(f"{path.name}: missing column: {c}")

    df["technique"]=tech
    df["execution"]=pd.to_numeric(df["execution"],errors="raise").astype(int)
    df["bertscore_f1"]=pd.to_numeric(df["bertscore_f1"],errors="coerce")

    config_key=(model,tech)
    if config_key in data_by_config:
        raise RuntimeError(f"Duplicate model×technique combination: {config_key}")

    invalid_cases=[]
    for cid,g in df.groupby("case_id"):
        if set(g["execution"].unique()) != EXPECTED_EXECUTIONS:
            invalid_cases.append(cid)

    inventory_rows.append({
        "modelo":model,"tecnica":tech,"arquivo":path.name,
        "linhas":len(df),"casos_unicos":df["case_id"].nunique(),
        "generation_ids_unicos":df["generation_id"].nunique(),
        "generation_ids_duplicados":int(df["generation_id"].duplicated().sum()),
        "casos_execucoes_invalidas":len(invalid_cases),
        "bertscore_ausentes":int(df["bertscore_f1"].isna().sum())
    })
    data_by_config[config_key]={"df":df,"arquivo":path.name}

inventory_df=pd.DataFrame(inventory_rows).sort_values(["modelo","tecnica"]).reset_index(drop=True)
display(inventory_df)

models=sorted(inventory_df["modelo"].unique())
problems=[]
if len(models)!=EXPECTED_MODEL_COUNT:
    problems.append(f"Expected 5 models; found {len(models)}")
if set(inventory_df["tecnica"].unique())!=set(EXPECTED_TECHNIQUES):
    problems.append("Technique set differs from the expected set")
if len(inventory_df)!=EXPECTED_BERTSCORE_FILES:
    problems.append(f"Expected 15 BERTScore files; found {len(inventory_df)}")

for m in models:
    for t in EXPECTED_TECHNIQUES:
        if (m,t) not in data_by_config:
            problems.append(f"Missing combination: {m} | {t}")

for _,r in inventory_df.iterrows():
    identifier=f"{r['modelo']} | {r['tecnica']}"
    if r["linhas"]!=EXPECTED_ROWS_PER_FILE: problems.append(f"{identifier}: rows={r['linhas']}")
    if r["casos_unicos"]!=EXPECTED_TOTAL_CASES: problems.append(f"{identifier}: cases={r['casos_unicos']}")
    if r["generation_ids_unicos"]!=EXPECTED_ROWS_PER_FILE: problems.append(f"{identifier}: unique IDs={r['generation_ids_unicos']}")
    if r["generation_ids_duplicados"]!=0: problems.append(f"{identifier}: duplicate IDs")
    if r["casos_execucoes_invalidas"]!=0: problems.append(f"{identifier}: cases without exact executions 1..10")
    if r["bertscore_ausentes"]!=0: problems.append(f"{identifier}: missing BERTScore")

# same case set
reference_case_ids=None
for config_key,content in data_by_config.items():
    s=set(content["df"]["case_id"])
    if reference_case_ids is None: reference_case_ids=s
    elif s!=reference_case_ids: problems.append(f"{config_key}: case_id set differs")

if problems:
    print("PROBLEMS:")
    for p in problems: print(" -",p)
    if STRICT_VALIDATION: raise RuntimeError("Validation failed.")
else:
    print(f"OK: {len(models)} models × 3 techniques = {len(inventory_df)} files; 259 cases × 10 executions.")
if ignored_files:
    print("Ignored because bertscore_f1 is missing:", ignored_files)

## 4. Consolidate and Aggregate the 10 Executions


In [ ]:
raw_parts=[]
for (m,t),obj in data_by_config.items():
    x=obj["df"][["case_id","execution","generation_id","bertscore_f1"]].copy()
    x["model"]=m; x["technique"]=t
    raw_parts.append(x)

raw_df=pd.concat(raw_parts,ignore_index=True)

aggregated_df=(
    raw_df.groupby(["case_id","model","technique"],as_index=False)
    .agg(
        bertscore_mean=("bertscore_f1","mean"),
        bertscore_median=("bertscore_f1","median"),
        bertscore_sd=("bertscore_f1","std"),
        bertscore_min=("bertscore_f1","min"),
        bertscore_max=("bertscore_f1","max"),
        execucoes=("execution","nunique")
    )
)
aggregated_df["bertscore_cv_percent"]=np.where(
    aggregated_df["bertscore_mean"]!=0,
    aggregated_df["bertscore_sd"]/np.abs(aggregated_df["bertscore_mean"])*100,
    np.nan
)

expected_aggregated_rows=EXPECTED_TOTAL_CASES*len(models)*len(EXPECTED_TECHNIQUES)
assert len(aggregated_df)==expected_aggregated_rows
assert (aggregated_df["execucoes"]==10).all()

mn=float(raw_df["bertscore_f1"].min()); mx=float(raw_df["bertscore_f1"].max())
BERT_SCALE="0–1 (approximate)" if mx<=1.5 else ("0–100 (approximate)" if mx<=100.5 else "unrecognized")
print(f"Raw records: {len(raw_df):,}")
print(f"Aggregated observations: {len(aggregated_df):,}")
print(f"Detected scale: {BERT_SCALE}; min={mn:.6f}; max={mx:.6f}")
display(aggregated_df.head())

## 5. Quality and Stability Descriptive Statistics


In [ ]:
descriptive_df=(
    aggregated_df.groupby(["model","technique"],as_index=False)
    .agg(
        n_casos=("case_id","nunique"),
        media_bertscore=("bertscore_mean","mean"),
        mediana_bertscore=("bertscore_mean","median"),
        sd_entre_casos=("bertscore_mean","std"),
        media_cv_execucoes=("bertscore_cv_percent","mean"),
        mediana_cv_execucoes=("bertscore_cv_percent","median"),
        media_sd_execucoes=("bertscore_sd","mean")
    )
)
descriptive_df["sem"]=descriptive_df["sd_entre_casos"]/np.sqrt(descriptive_df["n_casos"])
descriptive_df["ic95_media_inf"]=descriptive_df["media_bertscore"]-1.96*descriptive_df["sem"]
descriptive_df["ic95_media_sup"]=descriptive_df["media_bertscore"]+1.96*descriptive_df["sem"]
display(descriptive_df.sort_values(["model","media_bertscore"],ascending=[True,False]))

## 6. Normality Diagnosis of Paired Differences


In [ ]:
normality_rows=[]
for m in models:
    p=aggregated_df[aggregated_df["model"]==m].pivot(index="case_id",columns="technique",values="bertscore_mean")[EXPECTED_TECHNIQUES].dropna()
    for a,b in combinations(EXPECTED_TECHNIQUES,2):
        sh=safe_shapiro(p[a]-p[b])
        normality_rows.append({"modelo":m,"tecnica_a":a,"tecnica_b":b,"n":sh["n"],"W":sh["W"],"p":sh["p"],"rejeita_normalidade":sh["rejeita"]})
difference_normality_df=pd.DataFrame(normality_rows)
display(difference_normality_df)

## 7. Comparison of Techniques Within Each Model

The main test is **Friedman**, because the same 259 cases appear under all three techniques.  
When there is a global difference, pairs are compared using **paired Wilcoxon + Holm**.


In [ ]:
friedman_rows=[]; technique_posthoc_rows=[]
for m in models:
    p=aggregated_df[aggregated_df["model"]==m].pivot(index="case_id",columns="technique",values="bertscore_mean")[EXPECTED_TECHNIQUES].dropna()
    fr=stats.friedmanchisquare(*(p[t] for t in EXPECTED_TECHNIQUES))
    n=len(p); k=3
    friedman_rows.append({"modelo":m,"n":n,"friedman_chi2":float(fr.statistic),"df":2,"p_value":float(fr.pvalue),"kendall_W":float(fr.statistic/(n*(k-1))),"significativo":bool(fr.pvalue<ALPHA)})

    tmp=[]
    for j,(a,b) in enumerate(combinations(EXPECTED_TECHNIQUES,2)):
        va=p[a].to_numpy(float); vb=p[b].to_numpy(float)
        if np.allclose(va-vb,0):
            w,pv=0.0,1.0
        else:
            rr=stats.wilcoxon(va,vb,alternative="two-sided",zero_method="wilcox")
            w,pv=float(rr.statistic),float(rr.pvalue)
        d,lo,hi=bootstrap_mean_diff(va,vb,seed=BOOTSTRAP_SEED+j)
        tmp.append({"modelo":m,"tecnica_a":a,"tecnica_b":b,"n":len(va),"media_a":va.mean(),"media_b":vb.mean(),
                    "diferenca_media_a_menos_b":d,"ic95_diff_inf":lo,"ic95_diff_sup":hi,
                    "wilcoxon_W":w,"p_original":pv,"rank_biserial":paired_rank_biserial(va,vb)})
    adj=holm_bonferroni([x["p_original"] for x in tmp])
    for x,ph in zip(tmp,adj):
        x["p_holm"]=float(ph); x["significativo_holm"]=bool(ph<ALPHA)
        if not x["significativo_holm"]: x["interpretacao"]="diferença não significativa"
        elif x["diferenca_media_a_menos_b"]>0: x["interpretacao"]=f"{x['tecnica_a']} > {x['tecnica_b']}"
        else: x["interpretacao"]=f"{x['tecnica_b']} > {x['tecnica_a']}"
        technique_posthoc_rows.append(x)

model_friedman_df=pd.DataFrame(friedman_rows)
technique_posthoc_df=pd.DataFrame(technique_posthoc_rows)
display(model_friedman_df)
display(technique_posthoc_df)

## 8. Select the Best Technique for Each Model


In [ ]:
best_technique_rows=[]
for m in models:
    d=descriptive_df[descriptive_df["model"]==m].sort_values("media_bertscore",ascending=False).reset_index(drop=True)
    top=d.iloc[0]; tech=top["technique"]
    pairs=technique_posthoc_df[(technique_posthoc_df["modelo"]==m)&((technique_posthoc_df["tecnica_a"]==tech)|(technique_posthoc_df["tecnica_b"]==tech))]
    is_superior=True; details=[]
    for _,r in pairs.iterrows():
        if r["tecnica_a"]==tech:
            other_technique=r["tecnica_b"]; delta=r["diferenca_media_a_menos_b"]
        else:
            other_technique=r["tecnica_a"]; delta=-r["diferenca_media_a_menos_b"]
        ok=bool(r["significativo_holm"] and delta>0)
        is_superior &= ok
        details.append(f"{tech} vs {other_technique}: Δ={delta:.6f}; p-Holm={r['p_holm']:.6g}; {'superior' if ok else 'não demonstrou superioridade'}")
    most_stable=d.sort_values("media_cv_execucoes").iloc[0]["technique"]
    best_technique_rows.append({
        "modelo":m,"melhor_tecnica_observada":tech,
        "media_bertscore":float(top["media_bertscore"]),
        "mediana_bertscore":float(top["mediana_bertscore"]),
        "media_cv_percent":float(top["media_cv_execucoes"]),
        "tecnica_mais_estavel":most_stable,
        "superior_a_todas_holm":bool(is_superior),
        "status":"vencedora estatisticamente suportada" if is_superior else "maior média observada; superioridade sobre todas não demonstrada",
        "detalhes_posthoc":" | ".join(details)
    })
best_techniques_df=pd.DataFrame(best_technique_rows)
display(best_techniques_df)

## 9. Best Global Technique


In [ ]:
# mean across the 5 models within each case to avoid artificially multiplying n
global_technique_case_df=(
    aggregated_df.groupby(["case_id","technique"],as_index=False)
    .agg(bertscore_mean_modelos=("bertscore_mean","mean"))
)
pg=global_technique_case_df.pivot(index="case_id",columns="technique",values="bertscore_mean_modelos")[EXPECTED_TECHNIQUES].dropna()
frg=stats.friedmanchisquare(*(pg[t] for t in EXPECTED_TECHNIQUES))
global_kendall_w=float(frg.statistic/(len(pg)*(len(EXPECTED_TECHNIQUES)-1)))

global_technique_posthoc_rows=[]
for j,(a,b) in enumerate(combinations(EXPECTED_TECHNIQUES,2)):
    va=pg[a].to_numpy(float); vb=pg[b].to_numpy(float)
    rr=stats.wilcoxon(va,vb,alternative="two-sided",zero_method="wilcox")
    d,lo,hi=bootstrap_mean_diff(va,vb,seed=BOOTSTRAP_SEED+100+j)
    global_technique_posthoc_rows.append({"tecnica_a":a,"tecnica_b":b,"n":len(va),"media_a":va.mean(),"media_b":vb.mean(),"diferenca_media_a_menos_b":d,
                  "ic95_diff_inf":lo,"ic95_diff_sup":hi,"p_original":float(rr.pvalue),"rank_biserial":paired_rank_biserial(va,vb)})
adj=holm_bonferroni([x["p_original"] for x in global_technique_posthoc_rows])
for x,ph in zip(global_technique_posthoc_rows,adj):
    x["p_holm"]=float(ph); x["significativo_holm"]=bool(ph<ALPHA)
    if not x["significativo_holm"]: x["interpretacao"]="diferença não significativa"
    elif x["diferenca_media_a_menos_b"]>0: x["interpretacao"]=f"{x['tecnica_a']} > {x['tecnica_b']}"
    else: x["interpretacao"]=f"{x['tecnica_b']} > {x['tecnica_a']}"
global_technique_posthoc_df=pd.DataFrame(global_technique_posthoc_rows)

global_technique_means_df=(
    global_technique_case_df.groupby("technique",as_index=False)
    .agg(media_bertscore=("bertscore_mean_modelos","mean"),mediana_bertscore=("bertscore_mean_modelos","median"))
    .sort_values("media_bertscore",ascending=False).reset_index(drop=True)
)
BEST_GLOBAL_TECHNIQUE=global_technique_means_df.iloc[0]["technique"]
print(f"Friedman global: χ²={frg.statistic:.6f}, p={frg.pvalue:.6g}, Kendall W={global_kendall_w:.6f}")
display(global_technique_means_df)
display(global_technique_posthoc_df)
print("Global technique with the highest observed mean:",BEST_GLOBAL_TECHNIQUE)

## 10. Mixed Linear Model — Model × Technique Interaction


In [ ]:
LMM_OK=False; LMM_ERROR=None
lmm_tests_df=pd.DataFrame()

if STATSMODELS_AVAILABLE:
    try:
        z=aggregated_df[["case_id","model","technique","bertscore_mean"]].copy()
        full=smf.mixedlm("bertscore_mean ~ C(model) * C(technique)",z,groups=z["case_id"]).fit(reml=False,method="lbfgs",maxiter=500,disp=False)
        add=smf.mixedlm("bertscore_mean ~ C(model) + C(technique)",z,groups=z["case_id"]).fit(reml=False,method="lbfgs",maxiter=500,disp=False)
        only_m=smf.mixedlm("bertscore_mean ~ C(model)",z,groups=z["case_id"]).fit(reml=False,method="lbfgs",maxiter=500,disp=False)
        only_t=smf.mixedlm("bertscore_mean ~ C(technique)",z,groups=z["case_id"]).fit(reml=False,method="lbfgs",maxiter=500,disp=False)

        def likelihood_ratio_test(larger_model,smaller_model,effect_name):
            lr=2*(larger_model.llf-smaller_model.llf)
            dfd=int(round(larger_model.df_modelwc-smaller_model.df_modelwc))
            pv=float(stats.chi2.sf(lr,dfd))
            return {"efeito":effect_name,"LR":float(lr),"df":dfd,"p_value":pv,"significativo":bool(pv<ALPHA)}

        lmm_tests_df=pd.DataFrame([
            likelihood_ratio_test(full,add,"modelo × técnica"),
            likelihood_ratio_test(add,only_m,"técnica"),
            likelihood_ratio_test(add,only_t,"modelo")
        ])
        LMM_OK=True
        display(lmm_tests_df)
        print(full.summary())
    except Exception as e:
        LMM_ERROR=str(e)
        print("LMM did not converge/fit:",LMM_ERROR)
else:
    LMM_ERROR="statsmodels unavailable"
    print(LMM_ERROR)

## 11. Compare Models Using Each Model's Best Observed Technique


In [ ]:
best_config_parts=[]
for _,r in best_techniques_df.iterrows():
    x=aggregated_df[(aggregated_df["model"]==r["modelo"])&(aggregated_df["technique"]==r["melhor_tecnica_observada"])][["case_id","bertscore_mean"]].copy()
    x["model"]=r["modelo"]; x["selected_technique"]=r["melhor_tecnica_observada"]; best_config_parts.append(x)
best_config_df=pd.concat(best_config_parts,ignore_index=True)

pb=best_config_df.pivot(index="case_id",columns="model",values="bertscore_mean")[models].dropna()
frb=stats.friedmanchisquare(*(pb[m] for m in models))
best_config_kendall_w=float(frb.statistic/(len(pb)*(len(models)-1)))

best_config_model_posthoc_rows=[]
for j,(a,b) in enumerate(combinations(models,2)):
    va=pb[a].to_numpy(float); vb=pb[b].to_numpy(float)
    rr=stats.wilcoxon(va,vb,alternative="two-sided",zero_method="wilcox")
    d,lo,hi=bootstrap_mean_diff(va,vb,seed=BOOTSTRAP_SEED+1000+j)
    best_config_model_posthoc_rows.append({"modelo_a":a,"modelo_b":b,"n":len(va),"media_a":va.mean(),"media_b":vb.mean(),"diferenca_media_a_menos_b":d,
                  "ic95_diff_inf":lo,"ic95_diff_sup":hi,"p_original":float(rr.pvalue),"rank_biserial":paired_rank_biserial(va,vb)})
adj=holm_bonferroni([x["p_original"] for x in best_config_model_posthoc_rows])
for x,ph in zip(best_config_model_posthoc_rows,adj):
    x["p_holm"]=float(ph); x["significativo_holm"]=bool(ph<ALPHA)
    if not x["significativo_holm"]: x["interpretacao"]="diferença não significativa"
    elif x["diferenca_media_a_menos_b"]>0: x["interpretacao"]=f"{x['modelo_a']} > {x['modelo_b']}"
    else: x["interpretacao"]=f"{x['modelo_b']} > {x['modelo_a']}"
best_config_model_posthoc_df=pd.DataFrame(best_config_model_posthoc_rows)

best_config_model_ranking_df=(
    best_config_df.groupby(["model","selected_technique"],as_index=False)
    .agg(media_bertscore=("bertscore_mean","mean"),mediana_bertscore=("bertscore_mean","median"),sd=("bertscore_mean","std"))
    .sort_values("media_bertscore",ascending=False).reset_index(drop=True)
)
best_config_model_ranking_df.insert(0,"posicao",np.arange(1,len(best_config_model_ranking_df)+1))

print(f"Friedman for best configurations: χ²={frb.statistic:.6f}, p={frb.pvalue:.6g}, Kendall W={best_config_kendall_w:.6f}")
display(best_config_model_ranking_df)
display(best_config_model_posthoc_df)

## 12. Compare Models Under the Same Global Technique


In [ ]:
dg=aggregated_df[aggregated_df["technique"]==BEST_GLOBAL_TECHNIQUE].copy()
pm=dg.pivot(index="case_id",columns="model",values="bertscore_mean")[models].dropna()
frm=stats.friedmanchisquare(*(pm[m] for m in models))
global_model_kendall_w=float(frm.statistic/(len(pm)*(len(models)-1)))

global_model_posthoc_rows=[]
for j,(a,b) in enumerate(combinations(models,2)):
    va=pm[a].to_numpy(float); vb=pm[b].to_numpy(float)
    rr=stats.wilcoxon(va,vb,alternative="two-sided",zero_method="wilcox")
    d,lo,hi=bootstrap_mean_diff(va,vb,seed=BOOTSTRAP_SEED+2000+j)
    global_model_posthoc_rows.append({"modelo_a":a,"modelo_b":b,"n":len(va),"media_a":va.mean(),"media_b":vb.mean(),"diferenca_media_a_menos_b":d,
                   "ic95_diff_inf":lo,"ic95_diff_sup":hi,"p_original":float(rr.pvalue),"rank_biserial":paired_rank_biserial(va,vb)})
adj=holm_bonferroni([x["p_original"] for x in global_model_posthoc_rows])
for x,ph in zip(global_model_posthoc_rows,adj):
    x["p_holm"]=float(ph); x["significativo_holm"]=bool(ph<ALPHA)
    if not x["significativo_holm"]: x["interpretacao"]="diferença não significativa"
    elif x["diferenca_media_a_menos_b"]>0: x["interpretacao"]=f"{x['modelo_a']} > {x['modelo_b']}"
    else: x["interpretacao"]=f"{x['modelo_b']} > {x['modelo_a']}"
global_model_posthoc_df=pd.DataFrame(global_model_posthoc_rows)

global_model_ranking_df=(
    dg.groupby("model",as_index=False)
    .agg(media_bertscore=("bertscore_mean","mean"),mediana_bertscore=("bertscore_mean","median"),sd=("bertscore_mean","std"))
    .sort_values("media_bertscore",ascending=False).reset_index(drop=True)
)
global_model_ranking_df.insert(0,"posicao",np.arange(1,len(global_model_ranking_df)+1))
global_model_ranking_df["tecnica_global"]=BEST_GLOBAL_TECHNIQUE

print(f"Friedman modelos sob {BEST_GLOBAL_TECHNIQUE}: χ²={frm.statistic:.6f}, p={frm.pvalue:.6g}, Kendall W={global_model_kendall_w:.6f}")
display(global_model_ranking_df)
display(global_model_posthoc_df)

## 13. Quality and Stability Plots


In [ ]:
for m in models:
    d=descriptive_df[descriptive_df["model"]==m].set_index("technique").reindex(EXPECTED_TECHNIQUES).reset_index()
    plt.figure(figsize=(8,5))
    plt.bar(d["technique"],d["media_bertscore"],yerr=1.96*d["sem"],capsize=4)
    plt.ylabel("Mean BERTScore F1"); plt.xlabel("Technique"); plt.title(f"Quality — {m}")
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(8,5))
    plt.bar(d["technique"],d["media_cv_execucoes"])
    plt.ylabel("Mean CV across the 10 executions (%)"); plt.xlabel("Technique"); plt.title(f"Stability — {m}")
    plt.tight_layout(); plt.show()

## 14. Generate the Final TXT Summary


In [ ]:
summary_lines = []

def add_line(value=""):
    summary_lines.append(str(value))

add_line("=" * 95)
add_line("FINAL SUMMARY — BERTSCORE F1: TECHNIQUES AND MODELS")
add_line("=" * 95)
add_line()

add_line("1. DESIGN")
add_line("-" * 95)
add_line(f"Models: {len(models)}")
for m in models:
    add_line(f"  - {m}")
add_line(f"Techniques: {', '.join(EXPECTED_TECHNIQUES)}")
add_line(f"Cases per condition: {EXPECTED_TOTAL_CASES}")
add_line(f"Executions per case: {EXPECTED_TOTAL_EXECUTIONS}")
add_line(f"BERTScore files: {len(inventory_df)}")
add_line(f"Raw records: {len(raw_df):,}")
add_line(f"Aggregated observations: {len(aggregated_df):,}")
add_line(f"BERTScore scale: {BERT_SCALE}")
add_line()

add_line("2. BEST TECHNIQUE BY MODEL")
add_line("-" * 95)
for _, r in best_techniques_df.sort_values("modelo").iterrows():
    add_line(f"{r['modelo']}")
    add_line(f"  Best observed technique: {r['melhor_tecnica_observada']}")
    add_line(f"  Mean BERTScore: {r['media_bertscore']:.6f}")
    add_line(f"  Mean CV: {r['media_cv_percent']:.4f}%")
    add_line(f"  Most stable technique: {r['tecnica_mais_estavel']}")
    add_line(f"  Status: {r['status']}")
    add_line(f"  Post-hoc: {r['detalhes_posthoc']}")
    add_line()

add_line("3. FRIEDMAN BY MODEL")
add_line("-" * 95)
for _, r in model_friedman_df.iterrows():
    add_line(
        f"{r['modelo']}: chi2={r['friedman_chi2']:.6f}; "
        f"p={r['p_value']:.6g}; Kendall W={r['kendall_W']:.6f}; "
        f"{'significant' if r['significativo'] else 'not significant'}"
    )
add_line()

add_line("4. GLOBAL TECHNIQUE")
add_line("-" * 95)
add_line(f"Highest global mean: {BEST_GLOBAL_TECHNIQUE}")
add_line(
    f"Global Friedman: chi2={frg.statistic:.6f}; "
    f"p={frg.pvalue:.6g}; Kendall W={global_kendall_w:.6f}"
)
for _, r in global_technique_posthoc_df.iterrows():
    add_line(
        f"{r['tecnica_a']} vs {r['tecnica_b']}: "
        f"Δ={r['diferenca_media_a_menos_b']:.6f}; "
        f"95% CI=[{r['ic95_diff_inf']:.6f},{r['ic95_diff_sup']:.6f}]; "
        f"p-Holm={r['p_holm']:.6g}; {r['interpretacao']}"
    )
add_line()

add_line("5. MIXED LINEAR MODEL — MODEL × TECHNIQUE")
add_line("-" * 95)
if LMM_OK:
    for _, r in lmm_tests_df.iterrows():
        add_line(
            f"{r['efeito']}: LR={r['LR']:.6f}; df={int(r['df'])}; "
            f"p={r['p_value']:.6g}; "
            f"{'significant' if r['significativo'] else 'not significant'}"
        )
    interaction = bool(
        lmm_tests_df.loc[
            lmm_tests_df["efeito"] == "modelo × técnica",
            "significativo"
        ].iloc[0]
    )
    if interaction:
        add_line(
            "Interpretation: significant interaction; the best technique depends on the model. "
            "Prioritize model-specific conclusions."
        )
    else:
        add_line(
            "Interpretation: non-significant interaction; the global technique is a more "
            "plausible summary across models."
        )
else:
    add_line(f"LMM unavailable/non-convergent: {LMM_ERROR}")
add_line()

add_line("6. MODEL COMPARISON — EACH MODEL'S BEST CONFIGURATION")
add_line("-" * 95)
add_line(
    f"Friedman: chi2={frb.statistic:.6f}; "
    f"p={frb.pvalue:.6g}; Kendall W={best_config_kendall_w:.6f}"
)
for _, r in best_config_model_ranking_df.iterrows():
    add_line(
        f"{int(r['posicao'])}º {r['model']} ({r['selected_technique']}): "
        f"mean BERTScore={r['media_bertscore']:.6f}"
    )
add_line("Post-hoc:")
for _, r in best_config_model_posthoc_df.iterrows():
    add_line(
        f"{r['modelo_a']} vs {r['modelo_b']}: "
        f"Δ={r['diferenca_media_a_menos_b']:.6f}; "
        f"95% CI=[{r['ic95_diff_inf']:.6f},{r['ic95_diff_sup']:.6f}]; "
        f"p-Holm={r['p_holm']:.6g}; {r['interpretacao']}"
    )
add_line()

add_line("7. MODEL COMPARISON — SAME GLOBAL TECHNIQUE")
add_line("-" * 95)
add_line(f"Technique: {BEST_GLOBAL_TECHNIQUE}")
add_line(
    f"Friedman: chi2={frm.statistic:.6f}; "
    f"p={frm.pvalue:.6g}; Kendall W={global_model_kendall_w:.6f}"
)
for _, r in global_model_ranking_df.iterrows():
    add_line(
        f"{int(r['posicao'])}º {r['model']}: "
        f"mean BERTScore={r['media_bertscore']:.6f}"
    )
add_line("Post-hoc:")
for _, r in global_model_posthoc_df.iterrows():
    add_line(
        f"{r['modelo_a']} vs {r['modelo_b']}: "
        f"Δ={r['diferenca_media_a_menos_b']:.6f}; "
        f"95% CI=[{r['ic95_diff_inf']:.6f},{r['ic95_diff_sup']:.6f}]; "
        f"p-Holm={r['p_holm']:.6g}; {r['interpretacao']}"
    )
add_line()

add_line("8. AUTOMATIC CONCLUSION")
add_line("-" * 95)
top_best = best_config_model_ranking_df.iloc[0]
top_global = global_model_ranking_df.iloc[0]
add_line(f"Global technique with the highest mean BERTScore: {BEST_GLOBAL_TECHNIQUE}.")
add_line(
    f"Best observed configuration: {top_best['model']} + {top_best['selected_technique']} "
    f"(mean={top_best['media_bertscore']:.6f})."
)
add_line(
    f"Best model under the global technique {BEST_GLOBAL_TECHNIQUE}: "
    f"{top_global['model']} (mean={top_global['media_bertscore']:.6f})."
)
add_line(
    "Quality is defined by mean BERTScore F1; stability is reported separately "
    "through the CV of the 10 executions."
)
add_line(
    "Executions were aggregated before inferential tests to avoid pseudoreplication."
)

summary_text = "\n".join(summary_lines)
SUMMARY_PATH = OUTPUT_DIR / "resumo_final_bertscore.txt"
SUMMARY_PATH.write_text(summary_text, encoding="utf-8")
print(summary_text)
print("\nSummary saved to:", SUMMARY_PATH.resolve())


## 15. Export Audit Tables


In [ ]:
inventory_df.to_csv(OUTPUT_DIR/"inventario_arquivos_bertscore.csv",index=False,encoding="utf-8-sig")
raw_df.to_csv(OUTPUT_DIR/"bertscore_execucoes_consolidadas.csv",index=False,encoding="utf-8-sig")
aggregated_df.to_csv(OUTPUT_DIR/"bertscore_agregado_10_execucoes.csv",index=False,encoding="utf-8-sig")
descriptive_df.to_csv(OUTPUT_DIR/"estatistica_descritiva_modelo_tecnica.csv",index=False,encoding="utf-8-sig")
difference_normality_df.to_csv(OUTPUT_DIR/"normalidade_diferencas_tecnicas.csv",index=False,encoding="utf-8-sig")
model_friedman_df.to_csv(OUTPUT_DIR/"friedman_tecnicas_por_modelo.csv",index=False,encoding="utf-8-sig")
technique_posthoc_df.to_csv(OUTPUT_DIR/"posthoc_tecnicas_por_modelo.csv",index=False,encoding="utf-8-sig")
best_techniques_df.to_csv(OUTPUT_DIR/"melhor_tecnica_por_modelo.csv",index=False,encoding="utf-8-sig")
global_technique_posthoc_df.to_csv(OUTPUT_DIR/"posthoc_tecnicas_global.csv",index=False,encoding="utf-8-sig")
global_technique_means_df.to_csv(OUTPUT_DIR/"ranking_tecnicas_global.csv",index=False,encoding="utf-8-sig")
if LMM_OK:
    lmm_tests_df.to_csv(OUTPUT_DIR/"modelo_linear_misto_testes.csv",index=False,encoding="utf-8-sig")
best_config_model_ranking_df.to_csv(OUTPUT_DIR/"ranking_modelos_melhor_configuracao.csv",index=False,encoding="utf-8-sig")
best_config_model_posthoc_df.to_csv(OUTPUT_DIR/"posthoc_modelos_melhor_configuracao.csv",index=False,encoding="utf-8-sig")
global_model_ranking_df.to_csv(OUTPUT_DIR/"ranking_modelos_tecnica_global.csv",index=False,encoding="utf-8-sig")
global_model_posthoc_df.to_csv(OUTPUT_DIR/"posthoc_modelos_tecnica_global.csv",index=False,encoding="utf-8-sig")

print("Generated files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -",p.name)

# How to Interpret the Output

The best technique for each model is selected by the highest mean BERTScore F1, but the notebook reports whether that lead is statistically supported by the Holm-adjusted post-hoc comparisons.

- `vencedora estatisticamente suportada`: highest mean and significantly better than the other two techniques;
- `maior média observada; superioridade sobre todas não demonstrada`: descriptively leads, but at least one comparison did not demonstrate superiority.

If the `modelo × técnica` interaction in the mixed linear model is significant, the best technique depends on the model. In that scenario, model-specific conclusions are more important than a single global technique.

For model comparison, the analysis under the **same global technique** is more conservative than comparing each model after already selecting its own best technique.

> The Portuguese categorical labels are intentionally preserved because they are part of the persisted output contract used by existing files and downstream analyses.
